# EmotionMirror — FER2013 Training on Colab GPU

Trains a facial emotion classifier (7 classes) on FER2013.  
Uses EfficientNet-B0 fine-tuning for ~68-70% accuracy.  
Full run takes ~30-40 minutes on a T4 GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install albumentations datasets scikit-learn matplotlib seaborn opencv-python pyyaml

## 3. Write Project Files

In [ ]:
import os
os.makedirs('model/data', exist_ok=True)
os.makedirs('model/configs', exist_ok=True)
os.makedirs('model/checkpoints', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
print('Directories created.')

In [ ]:
%%writefile model/__init__.py


In [ ]:
%%writefile model/data/__init__.py


In [ ]:
%%writefile model/model.py
import torch
import torch.nn as nn
import torchvision.models as models
from typing import Optional


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool=True, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CustomCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32, pool=True, dropout=0.1),
            ConvBlock(32, 64, pool=True, dropout=0.1),
            ConvBlock(64, 128, pool=True, dropout=0.2),
            ConvBlock(128, 256, pool=True, dropout=0.2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.classifier(self.features(x))

    def get_feature_layer(self):
        return self.features[-1].block[-3]


class EfficientNetEmotion(nn.Module):
    def __init__(self, num_classes=7, pretrained=True, dropout=0.4):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.efficientnet_b0(weights=weights)
        self.features = base.features
        self.avgpool = base.avgpool
        in_features = base.classifier[1].in_features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )
        self._freeze_backbone()

    def _freeze_backbone(self):
        for param in self.features.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.features.parameters():
            param.requires_grad = True
        print('[Model] Backbone unfrozen — full fine-tuning enabled')

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def get_feature_layer(self):
        return self.features[-1]


def build_model(cfg):
    model_cfg = cfg['model']
    num_classes = cfg['data']['num_classes']
    architecture = model_cfg['architecture']

    if architecture == 'custom_cnn':
        model = CustomCNN(num_classes=num_classes, dropout=model_cfg['dropout'])
        print(f"[Model] CustomCNN: {sum(p.numel() for p in model.parameters()):,} params")
    elif architecture == 'efficientnet_b0':
        model = EfficientNetEmotion(
            num_classes=num_classes,
            pretrained=model_cfg.get('pretrained', True),
            dropout=model_cfg['dropout'],
        )
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[Model] EfficientNet-B0: {total:,} total params, {trainable:,} trainable")
    else:
        raise ValueError(f'Unknown architecture: {architecture}')

    return model

In [ ]:
%%writefile model/data/dataset.py
import os
from pathlib import Path
from typing import Optional, Tuple, List

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
EMOTION_TO_IDX = {e: i for i, e in enumerate(EMOTIONS)}


def get_transforms(split, image_size, normalize_mean, normalize_std):
    if split == 'train':
        return A.Compose([
            A.Resize(image_size, image_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.OneOf([
                A.GaussNoise(p=1.0),
                A.GaussianBlur(blur_limit=3, p=1.0),
            ], p=0.3),
            A.CoarseDropout(
                num_holes_range=(1, 8), hole_height_range=(4, 8),
                hole_width_range=(4, 8), fill=0, p=0.3
            ),
            A.Normalize(mean=normalize_mean, std=normalize_std),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=normalize_mean, std=normalize_std),
            ToTensorV2(),
        ])


class FER2013Dataset(Dataset):
    def __init__(self, data_dir, split, image_size=48,
                 normalize_mean=[0.507, 0.507, 0.507],
                 normalize_std=[0.255, 0.255, 0.255],
                 rgb=False):
        self.data_dir = Path(data_dir) / split
        self.split = split
        self.rgb = rgb
        self.transform = get_transforms(split, image_size, normalize_mean, normalize_std)

        self.samples = []
        for emotion in EMOTIONS:
            emotion_dir = self.data_dir / emotion
            if not emotion_dir.exists():
                raise FileNotFoundError(f'Missing directory: {emotion_dir}')
            for img_path in emotion_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    self.samples.append((img_path, EMOTION_TO_IDX[emotion]))

        if len(self.samples) == 0:
            raise RuntimeError(f'No images found in {self.data_dir}')

        print(f'[Dataset] {split}: {len(self.samples)} images across {len(EMOTIONS)} classes')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB' if self.rgb else 'L')
        img_array = np.array(img)
        if not self.rgb:
            img_array = np.stack([img_array] * 3, axis=-1)
        augmented = self.transform(image=img_array)
        return augmented['image'], label

    def get_class_weights(self):
        class_counts = torch.zeros(len(EMOTIONS))
        for _, label in self.samples:
            class_counts[label] += 1
        weights = 1.0 / class_counts
        return torch.tensor([weights[label] for _, label in self.samples])


def build_dataloaders(cfg):
    data_cfg = cfg['data']
    aug_cfg = cfg['augmentation']
    is_rgb = (data_cfg['image_size'] == 224)

    full_train = FER2013Dataset(
        data_dir=data_cfg['data_dir'], split='train',
        image_size=data_cfg['image_size'],
        normalize_mean=aug_cfg['normalize_mean'],
        normalize_std=aug_cfg['normalize_std'],
        rgb=is_rgb,
    )
    test_dataset = FER2013Dataset(
        data_dir=data_cfg['data_dir'], split='test',
        image_size=data_cfg['image_size'],
        normalize_mean=aug_cfg['normalize_mean'],
        normalize_std=aug_cfg['normalize_std'],
        rgb=is_rgb,
    )

    n_total = len(full_train)
    n_val = int(n_total * data_cfg['val_split'])
    n_train = n_total - n_val

    train_dataset, val_dataset = torch.utils.data.random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(42),
    )

    train_indices = train_dataset.indices
    all_weights = full_train.get_class_weights()
    train_weights = all_weights[train_indices]
    sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)

    batch_size = cfg['training']['batch_size']
    num_workers = min(2, os.cpu_count())

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    print(f'[DataLoader] train: {len(train_loader)} batches | '
          f'val: {len(val_loader)} batches | test: {len(test_loader)} batches')
    return train_loader, val_loader, test_loader

In [ ]:
%%writefile model/gradcam.py
from typing import Optional, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self._activations = None
        self._gradients = None
        self._forward_hook = target_layer.register_forward_hook(self._save_activations)
        self._backward_hook = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self._activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self._gradients = grad_output[0].detach()

    def __call__(self, image, target_class=None):
        self.model.eval()
        logits = self.model(image)
        probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
        pred_class = int(np.argmax(probs))
        if target_class is None:
            target_class = pred_class
        self.model.zero_grad()
        logits[0, target_class].backward()
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self._activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        h, w = image.shape[2], image.shape[3]
        cam = F.interpolate(cam, size=(h, w), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam_min, cam_max = cam.min(), cam.max()
        cam = (cam - cam_min) / (cam_max - cam_min) if cam_max > cam_min else np.zeros_like(cam)
        return cam, pred_class, probs

    def overlay(self, image_bgr, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
        if heatmap.shape != image_bgr.shape[:2]:
            heatmap = cv2.resize(heatmap, (image_bgr.shape[1], image_bgr.shape[0]))
        colored = cv2.applyColorMap(np.uint8(255 * heatmap), colormap)
        return cv2.addWeighted(image_bgr, 1 - alpha, colored, alpha, 0)

    def remove_hooks(self):
        self._forward_hook.remove()
        self._backward_hook.remove()

    def __del__(self):
        try:
            self.remove_hooks()
        except Exception:
            pass


def run_gradcam_on_batch(model, images, labels, num_samples=8, save_path=None):
    import matplotlib.pyplot as plt
    from model.data.dataset import EMOTIONS

    gradcam = GradCAM(model, target_layer=model.get_feature_layer())
    images = images[:num_samples]
    labels = labels[:num_samples]

    fig, axes = plt.subplots(2, num_samples // 2, figsize=(16, 6))
    axes = axes.flatten()

    for i, (img, label) in enumerate(zip(images, labels)):
        img_input = img.unsqueeze(0)
        heatmap, pred_class, probs = gradcam(img_input)
        img_display = img.permute(1, 2, 0).cpu().numpy()
        img_display = (img_display * 0.255 + 0.507).clip(0, 1)
        img_display_bgr = (img_display[:, :, ::-1] * 255).astype(np.uint8)
        overlay = gradcam.overlay(img_display_bgr, heatmap)
        axes[i].imshow(overlay[:, :, ::-1])
        gt, pred = EMOTIONS[label.item()], EMOTIONS[pred_class]
        axes[i].set_title(f'GT: {gt}\nPred: {pred} ({probs[pred_class]:.0%})',
                          color='green' if gt == pred else 'red', fontsize=8)
        axes[i].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    gradcam.remove_hooks()

In [ ]:
%%writefile model/train.py
import argparse
import os
import sys
import time
from pathlib import Path
from typing import Dict, Tuple

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from model.data.dataset import build_dataloaders, EMOTIONS
from model.model import build_model, EfficientNetEmotion


def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'[Device] GPU: {torch.cuda.get_device_name(0)}')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print('[Device] Apple MPS')
    else:
        device = torch.device('cpu')
        print('[Device] CPU')
    return device


def build_optimizer(model, cfg):
    train_cfg = cfg['training']
    if isinstance(model, EfficientNetEmotion) and 'backbone_lr_multiplier' in train_cfg:
        param_groups = [
            {'params': list(model.features.parameters()),
             'lr': train_cfg['learning_rate'] * train_cfg['backbone_lr_multiplier']},
            {'params': list(model.classifier.parameters()),
             'lr': train_cfg['learning_rate']},
        ]
    else:
        param_groups = model.parameters()

    name = train_cfg.get('optimizer', 'adam').lower()
    if name == 'adam':
        return optim.Adam(param_groups, lr=train_cfg['learning_rate'], weight_decay=train_cfg['weight_decay'])
    elif name == 'adamw':
        return optim.AdamW(param_groups, lr=train_cfg['learning_rate'], weight_decay=train_cfg['weight_decay'])
    elif name == 'sgd':
        return optim.SGD(param_groups, lr=train_cfg['learning_rate'], momentum=0.9, weight_decay=train_cfg['weight_decay'])
    raise ValueError(f'Unknown optimizer: {name}')


def build_scheduler(optimizer, cfg, train_loader_len):
    train_cfg = cfg['training']
    name = train_cfg.get('scheduler', 'cosine').lower()
    if name == 'cosine':
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_cfg['epochs'], eta_min=1e-6)
    elif name == 'one_cycle':
        return optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=train_cfg['learning_rate'],
            steps_per_epoch=train_loader_len, epochs=train_cfg['epochs'])
    elif name == 'step':
        return optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    return None


def train_epoch(model, loader, optimizer, criterion, device, scheduler=None,
                gradient_clip=1.0, scheduler_step_per_batch=False):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        if gradient_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        optimizer.step()
        if scheduler_step_per_batch and scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return {'loss': total_loss / len(loader), 'accuracy': correct / total}


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    return ({'loss': total_loss / len(loader), 'accuracy': (all_preds == all_labels).mean()},
            all_preds, all_labels)


def save_confusion_matrix(preds, labels, save_path):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[0])
    axes[0].set_title('Confusion matrix (counts)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=axes[1])
    axes[1].set_title('Confusion matrix (normalized)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()


def save_training_curves(history, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(epochs, history['train_acc'], label='train')
    axes[1].plot(epochs, history['val_acc'], label='val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    plt.savefig(save_path, dpi=100)
    plt.close()


def export_onnx(model, cfg, device):
    model.eval()
    image_size = cfg['data']['image_size']
    dummy_input = torch.randn(1, 3, image_size, image_size).to(device)
    onnx_path = cfg['output']['onnx_path']
    Path(onnx_path).parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        model.cpu(), dummy_input.cpu(), onnx_path,
        export_params=True, opset_version=17, do_constant_folding=True,
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    )
    print(f'[ONNX] Exported to {onnx_path}')


def train(cfg, resume=False):
    device = get_device()
    checkpoint_dir = Path(cfg['output']['checkpoint_dir'])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    exp_name = cfg['experiment_name']

    train_loader, val_loader, test_loader = build_dataloaders(cfg)
    model = build_model(cfg).to(device)

    label_smoothing = cfg['training'].get('label_smoothing', 0.0)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = build_optimizer(model, cfg)
    is_one_cycle = cfg['training'].get('scheduler', '') == 'one_cycle'
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))

    best_val_acc = 0.0
    patience_counter = 0
    start_epoch = 1
    patience = cfg['training']['early_stopping_patience']
    freeze_epochs = cfg['model'].get('freeze_backbone_epochs', 0)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    checkpoint_path = checkpoint_dir / f'{exp_name}_best.pt'
    if resume and checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        best_val_acc = ckpt['val_accuracy']
        start_epoch = ckpt['epoch'] + 1
        print(f"[Resume] Loaded checkpoint from epoch {ckpt['epoch']} (val acc {best_val_acc:.3f})")
        print(f'[Resume] Continuing from epoch {start_epoch}')

    print(f"\n{'='*60}")
    print(f"  Training: {exp_name}")
    print(f"  Epochs: {cfg['training']['epochs']} | Batch: {cfg['training']['batch_size']}")
    print(f"  Device: {device}")
    print(f"{'='*60}\n")

    start_time = time.time()

    for epoch in range(start_epoch, cfg['training']['epochs'] + 1):
        if epoch == freeze_epochs + 1 and isinstance(model, EfficientNetEmotion):
            model.unfreeze_backbone()
            optimizer = build_optimizer(model, cfg)
            scheduler = build_scheduler(optimizer, cfg, len(train_loader))

        train_metrics = train_epoch(
            model, train_loader, optimizer, criterion, device,
            scheduler=scheduler if is_one_cycle else None,
            gradient_clip=cfg['training'].get('gradient_clip', 1.0),
            scheduler_step_per_batch=is_one_cycle,
        )
        val_metrics, val_preds, val_labels = evaluate(model, val_loader, criterion, device)

        if scheduler is not None and not is_one_cycle:
            scheduler.step()

        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_acc'].append(val_metrics['accuracy'])

        elapsed = time.time() - start_time
        lr = optimizer.param_groups[-1]['lr']
        print(
            f"Epoch {epoch:3d}/{cfg['training']['epochs']} | "
            f"train loss: {train_metrics['loss']:.4f} acc: {train_metrics['accuracy']:.3f} | "
            f"val loss: {val_metrics['loss']:.4f} acc: {val_metrics['accuracy']:.3f} | "
            f"lr: {lr:.2e} | {elapsed/60:.1f}m"
        )

        if epoch % 5 == 0 and cfg['logging']['save_gradcam_samples']:
            try:
                from model.gradcam import run_gradcam_on_batch
                sample_images, sample_labels = next(iter(val_loader))
                gradcam_path = checkpoint_dir / f'{exp_name}_gradcam_epoch{epoch:03d}.png'
                run_gradcam_on_batch(model, sample_images, sample_labels, save_path=str(gradcam_path))
            except Exception as e:
                print(f'[GradCAM] Skipped: {e}')

        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_accuracy': best_val_acc,
                'config': cfg,
            }, checkpoint_path)
            print(f'  [best] New best! Saved to {checkpoint_path}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\n[EarlyStopping] No improvement for {patience} epochs. Stopping.')
                break

    # Final evaluation
    print(f"\n{'='*60}\n  Final evaluation on test set\n{'='*60}")
    ckpt = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    test_metrics, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    print(f'\nTest accuracy:     {test_metrics["accuracy"]:.4f}')
    print(f'Best val accuracy: {best_val_acc:.4f}')
    print()
    print(classification_report(test_labels, test_preds, target_names=EMOTIONS))

    if cfg['logging']['save_confusion_matrix']:
        save_confusion_matrix(test_preds, test_labels,
                              str(checkpoint_dir / f'{exp_name}_confusion_matrix.png'))
    save_training_curves(history, str(checkpoint_dir / f'{exp_name}_training_curves.png'))

    if cfg['output']['export_onnx']:
        export_onnx(model, cfg, device)

    print(f'\nTotal training time: {(time.time()-start_time)/60:.1f} minutes')
    return test_metrics['accuracy'], best_val_acc


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--resume', action='store_true')
    args = parser.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    train(cfg, resume=args.resume)

In [ ]:
%%writefile model/configs/efficientnet.yaml
experiment_name: efficientnet_b0_fer2013

model:
  architecture: efficientnet_b0
  dropout: 0.4
  pretrained: true
  freeze_backbone_epochs: 5

data:
  dataset: fer2013
  data_dir: model/data/fer2013
  num_classes: 7
  image_size: 224
  val_split: 0.1

augmentation:
  normalize_mean: [0.485, 0.456, 0.406]
  normalize_std: [0.229, 0.224, 0.225]

training:
  epochs: 40
  batch_size: 32
  learning_rate: 0.001
  backbone_lr_multiplier: 0.1
  weight_decay: 0.0001
  optimizer: adamw
  scheduler: cosine
  early_stopping_patience: 8
  gradient_clip: 1.0
  label_smoothing: 0.1

output:
  checkpoint_dir: model/checkpoints
  onnx_path: model/checkpoints/efficientnet_b0_fer2013.onnx
  export_onnx: true

logging:
  save_gradcam_samples: true
  save_confusion_matrix: true

In [ ]:
%%writefile scripts/download_hf.py
from pathlib import Path
import sys

LABEL_MAP = {0: 'angry', 1: 'disgust', 2: 'fear',
             3: 'happy', 4: 'sad', 5: 'surprise', 6: 'neutral'}
FER_DIR = Path('model/data/fer2013')


def download_split(split):
    from datasets import load_dataset
    import warnings
    warnings.filterwarnings('ignore')

    for emotion in LABEL_MAP.values():
        (FER_DIR / split / emotion).mkdir(parents=True, exist_ok=True)

    print(f'\n[{split}] Streaming from HuggingFace...')
    ds = load_dataset('clip-benchmark/wds_fer2013', split=split, streaming=True)

    counts = {e: 0 for e in LABEL_MAP.values()}
    for i, sample in enumerate(ds):
        emotion = LABEL_MAP[sample['cls']]
        img = sample['jpg']
        img.save(FER_DIR / split / emotion / f'{i:05d}.jpg', 'JPEG')
        counts[emotion] += 1
        if (i + 1) % 1000 == 0:
            print(f'  {sum(counts.values())} images saved...', end='\r')

    total = sum(counts.values())
    print(f'  {total} images saved.     ')
    for emotion, count in sorted(counts.items()):
        print(f'    {emotion:<10} {count:>5}  {"#" * (count // 200)}')
    return total


if __name__ == '__main__':
    if FER_DIR.exists() and (FER_DIR / 'train' / 'happy').exists():
        happy_count = len(list((FER_DIR / 'train' / 'happy').glob('*.jpg')))
        if happy_count > 100:
            print(f'Dataset already present ({happy_count} happy train images). Skipping.')
            sys.exit(0)

    train_total = download_split('train')
    test_total = download_split('test')
    print(f'\nDone. Train: {train_total}  Test: {test_total}')

## 4. Download FER2013 Dataset
Streams from HuggingFace — no Kaggle credentials needed. ~10-15 minutes.

In [ ]:
!python scripts/download_hf.py

In [ ]:
# Verify download
from pathlib import Path
fer_dir = Path('model/data/fer2013')
for split in ['train', 'test']:
    total = sum(len(list((fer_dir / split / e).glob('*.jpg')))
                for e in ['angry','disgust','fear','happy','neutral','sad','surprise'])
    print(f'{split}: {total} images')

## 5. Train EfficientNet-B0

Strategy:
- Epochs 1-5: backbone frozen, only the head trains (fast warm-up)
- Epochs 6-40: backbone unfrozen, full fine-tuning at 10x lower LR
- Expected: ~68-70% test accuracy in ~30-40 minutes on T4

In [ ]:
import sys
import yaml
import torch
import numpy as np

sys.path.insert(0, '.')

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

with open('model/configs/efficientnet.yaml') as f:
    cfg = yaml.safe_load(f)

from model.train import train
test_acc, best_val_acc = train(cfg)

## 6. View Results

In [ ]:
from IPython.display import Image, display
exp = cfg['experiment_name']
ckpt_dir = cfg['output']['checkpoint_dir']

print('Training curves:')
display(Image(f'{ckpt_dir}/{exp}_training_curves.png'))

In [ ]:
print('Confusion matrix:')
display(Image(f'{ckpt_dir}/{exp}_confusion_matrix.png'))

In [ ]:
# GradCAM — what the model looks at for each emotion
import glob
gradcam_files = sorted(glob.glob(f'{ckpt_dir}/{exp}_gradcam_*.png'))
if gradcam_files:
    print(f'Showing latest GradCAM ({gradcam_files[-1]}):')
    display(Image(gradcam_files[-1]))
else:
    print('No GradCAM images found (generated every 5 epochs during training).')

## 7. Download Checkpoint
Download the trained model to use in the EmotionMirror app.

In [ ]:
import os
from google.colab import files

ckpt_path = f"{cfg['output']['checkpoint_dir']}/{cfg['experiment_name']}_best.pt"
onnx_path = cfg['output'].get('onnx_path', '')

print(f'Checkpoint size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB')
files.download(ckpt_path)

if onnx_path and os.path.exists(onnx_path):
    print(f'ONNX size: {os.path.getsize(onnx_path) / 1e6:.1f} MB')
    files.download(onnx_path)

## 8. Copy Checkpoint to Backend (optional)

After downloading, place the checkpoint in your local project:
```
EmotionMirror/model/checkpoints/efficientnet_b0_fer2013_best.pt
```
Then restart the backend — it will auto-load the new model.